# Data Pipeline: Transcribe, Translate, Synthesize, Align

**GPU Budget:** ~6-8 hours on A10G or T4 (per direction)

This notebook runs the complete data preparation pipeline for creating
synthetic parallel speech pairs:

```
Source Audio  -->  Whisper/WhisperX (timestamps + text)  -->  NLLB translate
    -->  MMS-TTS synthesize target speech  -->  Contextual align
    -->  Silence insertion  -->  Mimi encode  -->  Save output
```

### Data Sources (choose one per run)
| SOURCE | Direction | Audio | Text source |
|--------|-----------|-------|-------------|
| `common_voice` | **Sw->En** | CV-Sw `clips/` MP3s | CV TSV `sentence` field (forced alignment for timestamps) |
| `kenspeech` | Sw->En | KenSpeech HF dataset | KenSpeech `transcript` field |
| `directory` | En->Sw | FLEURS en_us WAVs | Whisper free-form |

### Run Order
1. First run: **Sw->En** — set `SOURCE='common_voice'` (recommended) or `'kenspeech'`
2. Second run: **En->Sw** — set `SOURCE='directory'` (FLEURS)

**CV-Sw validated** (~70K utterances, ~150-200 hrs) is the largest source and produces
the best Hibiki training data. The TSV transcripts are used directly — Whisper is only
used for word-level timestamps via forced alignment, never for transcription.

In [ ]:
# Install dependencies
!pip install -q transformers accelerate torchaudio soundfile scipy faster-whisper sentencepiece

# whisperx provides forced-alignment for Common Voice sources:
#   aligns the *existing* CV sentence to audio at the word level using wav2vec2 phonemes.
#   Much more accurate than free-form Whisper for low-resource languages like Swahili.
#   Falls back gracefully to Whisper+initial_prompt if the Swahili alignment model
#   is unavailable — no manual fallback needed.
!pip install -q whisperx

# Note: bitsandbytes is intentionally NOT installed. We swapped MADLAD-400 for
# NLLB-200-distilled-1.3B (~2.6 GB fp16) which fits on a single T4 without
# quantization. int8/NF4 also produced degenerate outputs on T5 models in
# past runs, so we stay fp16 end-to-end.

In [ ]:
# HuggingFace authentication (fast downloads + push_to_hub uploads)
# Add your token in Kaggle: Settings > Secrets > Add secret named "HF_TOKEN"
import os

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token  # legacy env var fallback

    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("HuggingFace login successful")
except Exception as e:
    print(f"HF token not set ({e}). Downloads may be rate-limited and push_to_hub will not work.")

In [ ]:
import os
import sys
import torch
import gc

REPO_DIR = '/kaggle/input/datasets/victormugambi/hibiki-sw/hibiki-sw'
sys.path.insert(0, REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR

BASE_DIR = '/kaggle/working/hibiki-sw'
os.makedirs(BASE_DIR, exist_ok=True)

# === CHANGE THESE FOR EACH RUN ===
SOURCE_LANG = 'sw'
TARGET_LANG = 'en'

# === DATA SOURCE — pick ONE ===
# Option A (recommended): Common Voice Swahili validated split (~70K utterances)
# Pre-saved Kaggle dataset with clips/ + validated.tsv already extracted.
SOURCE = 'common_voice'
CV_DATASET_DIR = '/kaggle/input/datasets/victormugambi/cv-swahili/cv-validated'
CV_SPLIT = 'validated'

# Option B: KenSpeech (~5.8K samples)
# SOURCE = 'kenspeech'
# KENSPEECH_DIR = '/kaggle/input/kenspeech-sw'

# Option C: FLEURS / directory (En->Sw)
# SOURCE = 'directory'

# === T4 OPTIMIZATION SETTINGS ===
# With 70K samples, the full pipeline takes ~2-3 Kaggle sessions (9hr each).
# Every step resumes from where it left off — just re-run the notebook.
#
# Session plan:
#   Session 1: Steps 1-2 (transcribe + translate) — ~6-7 hrs
#   Session 2: Steps 3-4 (align + synthesize)     — ~8-9 hrs
#   Session 3: Step 5 (Mimi encode)                — ~3-4 hrs
#
# Set MAX_SAMPLES=1000 for a quick debug run, None for full dataset.
MAX_SAMPLES = None

# T4: 16GB VRAM. We use Whisper small (not medium) for timestamps —
# sufficient for forced alignment fallback and saves ~700MB VRAM.
WHISPER_MODEL = 'small'
WHISPER_COMPUTE = 'int8'  # int8 is faster and uses less VRAM on T4

# NLLB: batch_size 8 fits comfortably on T4 in fp16 (~2.6GB model)
NLLB_BATCH_SIZE = 8

def free_gpu():
    """Flush GPU memory between pipeline steps."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        used = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_mem / 1e9
        print(f'GPU memory: {used:.1f}GB / {total:.1f}GB')

print(f'Direction: {SOURCE_LANG} -> {TARGET_LANG}')
print(f'Source: {SOURCE}')
if SOURCE == 'common_voice':
    print(f'CV dataset dir: {CV_DATASET_DIR}')
    print(f'CV split: {CV_SPLIT}')
    print(f'  exists: {os.path.exists(CV_DATASET_DIR)}')
elif SOURCE == 'kenspeech':
    print(f'KenSpeech dir: {KENSPEECH_DIR} (exists={os.path.exists(KENSPEECH_DIR)})')
print(f'MAX_SAMPLES: {MAX_SAMPLES or "all"}')
print(f'Whisper: {WHISPER_MODEL} ({WHISPER_COMPUTE})')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
gpu_mem = f'{torch.cuda.get_device_properties(0).total_mem / 1e9:.0f}GB' if torch.cuda.is_available() else ''
print(f'GPU: {gpu_name} {gpu_mem}')

In [ ]:
# Verify data source accessibility
if SOURCE == 'common_voice':
    from data.prepare.local_cv_loader import CommonVoiceLocal

    cv = CommonVoiceLocal(dataset_dir=CV_DATASET_DIR, split=CV_SPLIT, load_audio=False)
    stats = cv.get_stats()
    print(f'Common Voice Swahili ({CV_SPLIT}): {stats["total_samples"]} utterances')
    print(f'  Unique speakers: {stats["unique_speakers"]}')
    print(f'  Avg sentence length: {stats["avg_sentence_length"]:.0f} chars')

    # Spot-check: verify a few clips exist
    import random
    sample_indices = random.sample(range(len(cv)), min(5, len(cv)))
    clips_dir = os.path.join(CV_DATASET_DIR, 'clips')
    missing = 0
    for idx in sample_indices:
        entry = cv.entries[idx]
        clip_path = os.path.join(clips_dir, entry.get('path', ''))
        if not os.path.exists(clip_path):
            missing += 1
    if missing > 0:
        print(f'  WARNING: {missing}/5 sampled clips not found in {clips_dir}')
        print(f'  Make sure clips/ directory contains the MP3 files.')
    else:
        print(f'  Clip files verified (5/5 spot-checked)')

    if MAX_SAMPLES:
        effective = min(MAX_SAMPLES, stats["total_samples"])
        print(f'  Will process: {effective} samples (MAX_SAMPLES={MAX_SAMPLES})')
    else:
        print(f'  Will process: all {stats["total_samples"]} samples')

    del cv
    print('Dataset ready!')

elif SOURCE == 'kenspeech':
    from data.prepare.kenspeech_loader import KenSpeechLoader
    ks = KenSpeechLoader(load_audio=False, local_dir=KENSPEECH_DIR)
    print(f'KenSpeech loaded: {len(ks)} samples')
    stats = ks.get_stats()
    print(f'Unique speakers: {stats["unique_speakers"]}')
    print(f'Avg sentence length: {stats["avg_sentence_length"]:.0f} chars')
    del ks
    print('Dataset ready!')

elif SOURCE == 'directory':
    FLEURS_AUDIO_DIR = f'{BASE_DIR}/fleurs_en_audio'
    if not os.path.exists(FLEURS_AUDIO_DIR) or len(os.listdir(FLEURS_AUDIO_DIR)) == 0:
        print('Extracting FLEURS en_us audio to WAV files...')
        os.makedirs(FLEURS_AUDIO_DIR, exist_ok=True)
        from datasets import load_dataset
        import soundfile as sf
        import numpy as np

        ds = load_dataset("google/fleurs", "en_us", split="train")
        for i, sample in enumerate(ds):
            audio = sample["audio"]
            wav_path = os.path.join(FLEURS_AUDIO_DIR, f"fleurs_en_{i:05d}.wav")
            sf.write(wav_path, np.array(audio["array"], dtype=np.float32), audio["sampling_rate"])
        print(f'Extracted {len(ds)} WAV files to {FLEURS_AUDIO_DIR}')
    else:
        n_wavs = len([f for f in os.listdir(FLEURS_AUDIO_DIR) if f.endswith('.wav')])
        print(f'FLEURS audio already extracted: {n_wavs} WAV files')
    print('Dataset ready!')

## Step 1: Whisper Transcription (~2-3 hrs for KenSpeech, ~1 hr for FLEURS)

### Timestamp strategy (word-level timings for silence insertion)

| Source | Text | Timestamps |
|--------|------|------------|
| **KenSpeech** (Sw→En) | KenSpeech `sentence` field (existing, high-quality) | Whisper guided by `initial_prompt=sentence` — produces timestamps anchored to the known text rather than free-form transcription |
| **FLEURS** (En→Sw) | Whisper free-form transcription | Whisper free-form |
| **Common Voice** (any) | CV `sentence` field — **never** Whisper output | **Tier 1** WhisperX forced alignment (wav2vec2 phoneme model, most accurate) → **Tier 2** Whisper + `initial_prompt=sentence` → **Tier 3** uniform fallback |

> **Why this matters:** Word timestamps flow through the entire pipeline — transcription → translation → TTS synthesis → silence insertion. Inaccurate timestamps (from free-form Whisper on low-resource Swahili) cause the silence insertion step to insert pauses at the wrong positions, corrupting the causal alignment. Using the human transcript as the ground truth eliminates transcription errors entirely; we only need Whisper/WhisperX to tell us *when* each word was spoken, not *what* was said.

> **CV sources:** Pass `--forced_alignment` (default: on) to `transcribe_whisper.py`. WhisperX loads `wav2vec2-large-xlsr-53-swahili` automatically. If that model is unavailable, falls back to Whisper+initial_prompt silently.

In [ ]:
# Restore transcriptions from saved dataset (skip Whisper if already done)
import shutil

TRANSCRIPTS_DATASET = f'/kaggle/input/datasets/victormugambi/transcriptions'
TRANSCRIPTION_DIR = f'{BASE_DIR}/transcriptions/{SOURCE_LANG}'

existing = len([f for f in os.listdir(TRANSCRIPTION_DIR) if f.endswith('.json')]) if os.path.exists(TRANSCRIPTION_DIR) else 0

if existing > 0:
    print(f'Transcriptions already present: {existing} files — skipping restore')
else:
    src = f'{TRANSCRIPTS_DATASET}/{SOURCE_LANG}'
    if os.path.isdir(src):
        os.makedirs(TRANSCRIPTION_DIR, exist_ok=True)
        shutil.copytree(src, TRANSCRIPTION_DIR, dirs_exist_ok=True)
        existing = len([f for f in os.listdir(TRANSCRIPTION_DIR) if f.endswith('.json')])
        print(f'Restored {existing} transcription files from dataset')
    else:
        print(f'No saved transcriptions at {src} — will run transcription in the next cell')

print(f'TRANSCRIPTION_DIR = {TRANSCRIPTION_DIR}')
print(f'Existing transcriptions: {existing}')

In [ ]:
TRANSCRIPTION_DIR = f'{BASE_DIR}/transcriptions/{SOURCE_LANG}'

existing = len([f for f in os.listdir(TRANSCRIPTION_DIR) if f.endswith('.json')]) if os.path.exists(TRANSCRIPTION_DIR) else 0
print(f'Existing transcriptions: {existing}')

max_samples_flag = f'--max_samples {MAX_SAMPLES}' if MAX_SAMPLES else ''

if SOURCE == 'common_voice':
    # CV-Sw: text from validated.tsv sentence column.
    # Timestamps via WhisperX forced alignment (wav2vec2, ~1.2GB).
    # Whisper only loaded if WhisperX fails — using small+int8 to save T4 VRAM.
    if existing > 0:
        print(f'Resuming from {existing} existing transcriptions...')
    !cd {REPO_DIR} && python data/prepare/transcribe_whisper.py \
        --source common_voice \
        --dataset_dir {CV_DATASET_DIR} \
        --lang {SOURCE_LANG} \
        --split {CV_SPLIT} \
        --output_dir {TRANSCRIPTION_DIR} \
        --whisper_model {WHISPER_MODEL} \
        --compute_type {WHISPER_COMPUTE} \
        --forced_alignment \
        {max_samples_flag} \
        --resume_from {existing}

elif SOURCE == 'kenspeech':
    if existing > 0:
        print(f'Transcriptions already present — skipping Whisper entirely.')
    else:
        !cd {REPO_DIR} && python data/prepare/transcribe_whisper.py \
            --source kenspeech \
            --kenspeech_dir {KENSPEECH_DIR} \
            --lang {SOURCE_LANG} \
            --output_dir {TRANSCRIPTION_DIR} \
            --whisper_model {WHISPER_MODEL} \
            --compute_type {WHISPER_COMPUTE} \
            {max_samples_flag} \
            --resume_from {existing}

else:
    !cd {REPO_DIR} && python data/prepare/transcribe_whisper.py \
        --source directory \
        --lang {SOURCE_LANG} \
        --audio_dir {FLEURS_AUDIO_DIR} \
        --output_dir {TRANSCRIPTION_DIR} \
        --whisper_model {WHISPER_MODEL} \
        --compute_type {WHISPER_COMPUTE} \
        {max_samples_flag}

# Free Whisper + WhisperX from GPU before next step
free_gpu()
print('Transcription step done.')

In [ ]:
# Verify transcriptions
import json
from pathlib import Path

trans_files = sorted(Path(TRANSCRIPTION_DIR).glob('*.json'))
print(f'Total transcription files: {len(trans_files)}')

# Preview a sample
if trans_files:
    with open(trans_files[0]) as f:
        sample = json.load(f)
    print(f'\nSample: {trans_files[0].name}')
    print(f'  Text: {sample.get("text", "")[:100]}')
    print(f'  Duration: {sample.get("audio_duration", 0):.1f}s')
    print(f'  Words: {len(sample.get("words", []))}')
    if sample.get('words'):
        print(f'  First word: {sample["words"][0]}')

## Step 2: NLLB Translation (~1-2 hrs)

Translate all transcriptions to the target language using NLLB-200-distilled-1.3B.

In [ ]:
import shutil

DIRECTION = f'{SOURCE_LANG}2{TARGET_LANG}'
TRANSLATION_DIR = f'{BASE_DIR}/translations/{DIRECTION}'
TRANSLATIONS_DATASET = '/kaggle/input/datasets/victormugambi/translations'

existing = len([f for f in os.listdir(TRANSLATION_DIR) if f.endswith('.json')]) if os.path.exists(TRANSLATION_DIR) else 0
print(f'Existing translations: {existing}')

# Restore from saved dataset if available
if existing == 0:
    src = f'{TRANSLATIONS_DATASET}/{DIRECTION}'
    if os.path.isdir(src):
        os.makedirs(TRANSLATION_DIR, exist_ok=True)
        shutil.copytree(src, TRANSLATION_DIR, dirs_exist_ok=True)
        existing = len([f for f in os.listdir(TRANSLATION_DIR) if f.endswith('.json')])
        print(f'Restored {existing} translation files from dataset')
    else:
        print(f'Saved translations not found at {src} — will run NLLB below.')

n_source = len(list(Path(TRANSCRIPTION_DIR).glob('*.json'))) if os.path.exists(TRANSCRIPTION_DIR) else 0
target_count = MAX_SAMPLES if MAX_SAMPLES else n_source

if existing >= target_count or (existing > 0 and existing >= n_source):
    print(f'Translations already present ({existing}) — skipping NLLB entirely.')
else:
    max_samples_flag = f'--max_samples {MAX_SAMPLES}' if MAX_SAMPLES else ''

    # Smoke test
    print('\n--- Smoke test (~1 min) ---')
    !cd {REPO_DIR} && python data/prepare/translate_nllb.py \
        --source_lang {SOURCE_LANG} \
        --target_lang {TARGET_LANG} \
        --dtype float16 \
        --device_map auto \
        --smoke_test

    # Full translation run — batch_size tuned for T4 (16GB)
    # NLLB-1.3B fp16 uses ~2.6GB; batch_size 8 leaves headroom for KV cache
    print('\n--- Full translation run ---')
    !cd {REPO_DIR} && python data/prepare/translate_nllb.py \
        --input_dir {TRANSCRIPTION_DIR} \
        --output_dir {TRANSLATION_DIR} \
        --source_lang {SOURCE_LANG} \
        --target_lang {TARGET_LANG} \
        --dtype float16 \
        --device_map auto \
        --batch_size {NLLB_BATCH_SIZE} \
        {max_samples_flag} \
        --resume_from {existing}

# Free NLLB from GPU before next step
free_gpu()
print('Translation step done.')

In [ ]:
# Verify translations — FAIL LOUDLY if nothing was produced
trans_files = sorted(Path(TRANSLATION_DIR).glob('*.json'))
print(f'Total translation files: {len(trans_files)}')

if len(trans_files) == 0:
    raise RuntimeError(
        f"TRANSLATION STEP PRODUCED 0 FILES in {TRANSLATION_DIR}\n"
        "Check cell 10 output for errors from translate_nllb.py.\n"
        "Possible causes:\n"
        "  - TRANSCRIPTION_DIR is empty (transcription step failed)\n"
        "  - NLLB model failed to load (OOM or HF download error)\n"
        "Do NOT continue — remaining steps have nothing to process."
    )

if trans_files:
    with open(trans_files[0]) as f:
        sample = json.load(f)
    print(f'\nSample: {trans_files[0].name}')
    print(f'  Source ({SOURCE_LANG}): {sample.get("source_text", "")[:100]}')
    print(f'  Target ({TARGET_LANG}): {sample.get("translated_text", "")[:100]}')
    if not sample.get("translated_text", "").strip():
        raise RuntimeError(
            "translated_text is EMPTY in the first translation file.\n"
            "Translation ran but produced blank outputs — check NLLB logs."
        )

## Step 3: Contextual Alignment (~30 min)

Compute per-word alignment between source and target text using
NLLB-200-distilled-1.3B perplexity (Hibiki paper, Section 3.2.1, eq. 6).
The paper used MADLAD-400-3B; we substituted NLLB because MADLAD's
encoder/decoder produced broken logits in our environment. The algorithm
is model-agnostic — any encoder-decoder MT with reliable log-probs works.

This determines which source word maps to which target word, which is
needed for the silence insertion step.

In [ ]:
ALIGNMENT_DIR = f'{BASE_DIR}/alignments/{DIRECTION}'

max_samples_flag = f'--max_samples {MAX_SAMPLES}' if MAX_SAMPLES else ''

# Contextual alignment loads NLLB again for perplexity scoring.
# On T4 this takes ~2.6GB VRAM — same as the translation step.
!cd {REPO_DIR} && python data/prepare/run_pipeline.py \
    --source {SOURCE} \
    --source_lang {SOURCE_LANG} \
    --target_lang {TARGET_LANG} \
    --base_dir {BASE_DIR} \
    {max_samples_flag} \
    --step align

# Free NLLB alignment model from GPU before synthesis step
free_gpu()
print('Alignment step done.')

In [ ]:
# Verify alignments
align_files = sorted(Path(ALIGNMENT_DIR).glob('*.json'))
print(f'Total alignment files: {len(align_files)}')

if align_files:
    with open(align_files[0]) as f:
        sample = json.load(f)
    print(f'\nSample: {align_files[0].name}')
    print(f'  Source: {sample.get("source_text", "")[:80]}')
    print(f'  Target: {sample.get("translated_text", "")[:80]}')
    print(f'  Alignment pairs: {len(sample.get("alignment", []))}')
    if sample.get('alignment'):
        print(f'  First 5 pairs: {sample["alignment"][:5]}')

## Step 4: TTS Synthesis + Silence Insertion (~2-3 hrs)

This is the key step that creates the synthetic parallel speech:
1. Synthesize target-language speech from translations using MMS-TTS
2. Get word timestamps on synthesized audio with Whisper
3. Apply silence insertion for causal alignment (Hibiki paper Section 3.2)

**For Sw->En:** Uses pretrained `facebook/mms-tts-eng`
**For En->Sw:** Uses pretrained `facebook/mms-tts-swh`

In [ ]:
SYNTH_DIR = f'{BASE_DIR}/synthetic_audio/{DIRECTION}'
max_samples_flag = f'--max_samples {MAX_SAMPLES}' if MAX_SAMPLES else ''

# MMS-TTS synthesis + Whisper timestamp extraction + silence insertion.
# On T4: MMS-TTS (~1GB) + Whisper medium (~1.5GB int8) ≈ 2.5GB VRAM.
#
# NOTE: 70K samples at ~2-3 samples/sec = ~7-10 hours.
# This will likely span TWO Kaggle sessions. The script is resumable —
# just re-run this cell in a new session and it picks up where it left off.
# Check the "Synthesized:" count in the output to track progress.

cmd = (
    f"cd {REPO_DIR} && python data/prepare/synthesize_tts.py"
    f" --translation_dir {TRANSLATION_DIR}"
    f" --output_dir {SYNTH_DIR}"
    f" --target_lang {TARGET_LANG}"
    f" --alignment_dir {ALIGNMENT_DIR}"
    f" --whisper_model {WHISPER_MODEL}"
    f" --target_sr 24000"
    f" {max_samples_flag}"
)

print(f'Running synthesis...\n{cmd}\n')
!{cmd}

# Free MMS-TTS + Whisper from GPU before encoding step
free_gpu()
print('Synthesis step done.')

In [ ]:
# Verify synthesized audio — FAIL LOUDLY if synthesis produced nothing
import IPython.display as ipd
import torchaudio

aligned_dir = f'{SYNTH_DIR}/aligned_audio'
raw_dir = f'{SYNTH_DIR}/raw_synthesis/wavs'

# Check which directory has output
if os.path.exists(aligned_dir) and len(list(Path(aligned_dir).glob('*.wav'))) > 0:
    wav_dir = aligned_dir
else:
    wav_dir = raw_dir

n_wavs = len(list(Path(wav_dir).glob('*.wav'))) if os.path.exists(wav_dir) else 0
print(f'Synthesized WAVs in {wav_dir}: {n_wavs} total')

if n_wavs == 0:
    # Print diagnostic info to help debug
    trans_files = list(Path(TRANSLATION_DIR).glob('*.json')) if os.path.exists(TRANSLATION_DIR) else []
    print(f'\nDiagnostic:')
    print(f'  Translation files: {len(trans_files)} in {TRANSLATION_DIR}')
    print(f'  SYNTH_DIR exists: {os.path.exists(SYNTH_DIR)}')
    raw_synthesis_dir = f'{SYNTH_DIR}/raw_synthesis'
    print(f'  raw_synthesis dir exists: {os.path.exists(raw_synthesis_dir)}')
    if os.path.exists(raw_synthesis_dir):
        print(f'  raw_synthesis contents: {os.listdir(raw_synthesis_dir)}')
    raise RuntimeError(
        f"SYNTHESIS PRODUCED 0 AUDIO FILES in {wav_dir}\n"
        "Check cell 15 output for errors from synthesize_tts.py.\n"
        "Possible causes:\n"
        "  - Translation dir was empty (0 JSON files to synthesize)\n"
        "  - MMS-TTS model failed to load (OOM or HF download error)\n"
        "  - All synthesis calls failed silently (check 'Failed: N' count in cell 15)\n"
        "Do NOT continue to encoding — there is nothing to encode."
    )

wav_files = sorted(Path(wav_dir).glob('*.wav'))[:5]
print()
for wav_path in wav_files:
    waveform, sr = torchaudio.load(str(wav_path))
    dur = waveform.shape[1] / sr
    print(f'{wav_path.name} ({dur:.2f}s, {sr}Hz)')
    ipd.display(ipd.Audio(waveform.squeeze().numpy(), rate=sr))

## Step 5: Encode Audio with Mimi Codec

Encode both source audio and synthesized target audio through the Mimi
neural codec to get discrete token representations.

Output: `.npy` files of shape `(Q=8, T)` -- 8 codebooks x T timesteps.

In [ ]:
AUDIO_TOKEN_DIR = f'{BASE_DIR}/audio_tokens'
max_samples_flag = f'--max_samples {MAX_SAMPLES}' if MAX_SAMPLES else ''

# Mimi codec encoding: loads kyutai/mimi (~300MB VRAM).
# Encoding is fast — ~10-20 samples/sec on T4.
# 70K samples ≈ 1-2 hours.

# Encode source audio
if SOURCE == 'common_voice':
    print('=== Encoding source audio (Common Voice Swahili) ===')
    !cd {REPO_DIR} && python data/prepare/encode_audio.py \
        --source common_voice \
        --lang {SOURCE_LANG} \
        --dataset_dir {CV_DATASET_DIR} \
        --split {CV_SPLIT} \
        --output_dir {AUDIO_TOKEN_DIR}/cv_sw \
        --num_codebooks 8 \
        {max_samples_flag} \
        --max_duration 20.0

elif SOURCE == 'kenspeech':
    print('=== Encoding source audio (KenSpeech Swahili) ===')
    !cd {REPO_DIR} && python data/prepare/encode_audio.py \
        --source kenspeech \
        --kenspeech_dir {KENSPEECH_DIR} \
        --output_dir {AUDIO_TOKEN_DIR}/kenspeech_sw \
        --num_codebooks 8 \
        {max_samples_flag} \
        --max_duration 20.0

else:
    print('=== Encoding source audio (FLEURS English) ===')
    !cd {REPO_DIR} && python data/prepare/encode_audio.py \
        --source directory \
        --audio_dir {FLEURS_AUDIO_DIR} \
        --output_dir {AUDIO_TOKEN_DIR}/fleurs_en \
        --num_codebooks 8 \
        --max_duration 20.0

# Free Mimi from GPU between source and target encoding
free_gpu()
print('Source encoding done.')

In [ ]:
# Encode synthesized target audio
synth_wav_dir = f'{SYNTH_DIR}/aligned_audio'
if not os.path.exists(synth_wav_dir):
    synth_wav_dir = f'{SYNTH_DIR}/raw_synthesis/wavs'

print(f'=== Encoding synthesized {TARGET_LANG} audio ===')
!cd {REPO_DIR} && python data/prepare/encode_audio.py \
    --source directory \
    --audio_dir {synth_wav_dir} \
    --output_dir {AUDIO_TOKEN_DIR}/synth_{TARGET_LANG} \
    --num_codebooks 8 \
    --max_duration 30.0

# Free Mimi — encoding is the last GPU step in the pipeline
free_gpu()
print('Target encoding done.')

In [ ]:
# ── KenSpeech token availability check (informational) ──────────────────────
# KenSpeech tokens are encoded in 00_data_preparation.ipynb and stored in
# a separate Kaggle dataset.  If you've attached that dataset, this cell will
# confirm how many tokens are available to fold into S2ST training manifests
# as additional Swahili source audio (for the Sw→En direction).
#
# No pipeline execution happens here — this is a reference check only.

KENSPEECH_TOKEN_DIR = f'{AUDIO_TOKEN_DIR}/kenspeech'  # from 00_data_preparation.ipynb

if SOURCE_LANG == 'sw':
    from pathlib import Path as _Path
    if os.path.isdir(KENSPEECH_TOKEN_DIR):
        n_ks = len(list(_Path(KENSPEECH_TOKEN_DIR).glob('*.npy')))
        print(f"KenSpeech tokens available: {n_ks} .npy files in {KENSPEECH_TOKEN_DIR}")
        print(
            "\nTo include KenSpeech as additional Sw source audio in the S2ST manifest,\n"
            "pass --source_token_dirs to create_s2st_manifest.py:\n"
            f"  --source_token_dirs {AUDIO_TOKEN_DIR}/kenspeech_sw {KENSPEECH_TOKEN_DIR}\n"
            "Both directories will be scanned for .npy files and interleaved in the manifest."
        )
    else:
        print(
            f"KenSpeech token directory not found at {KENSPEECH_TOKEN_DIR}.\n"
            "Run 00_data_preparation.ipynb first (cell 9) to encode KenSpeech audio,\n"
            "then attach that Kaggle dataset to this notebook."
        )
else:
    print(f"SOURCE_LANG={SOURCE_LANG!r} — KenSpeech Sw tokens not needed for En→Sw direction.")

In [ ]:
# Verify encoded tokens
import numpy as np

source_subdir_map = {
    'common_voice': 'cv_sw',
    'kenspeech': 'kenspeech_sw',
    'directory': f'fleurs_{SOURCE_LANG}',
}
source_subdir = source_subdir_map.get(SOURCE, f'fleurs_{SOURCE_LANG}')

for subdir in [source_subdir, f'synth_{TARGET_LANG}']:
    token_dir = f'{AUDIO_TOKEN_DIR}/{subdir}'
    if os.path.exists(token_dir):
        npy_files = list(Path(token_dir).glob('*.npy'))
        print(f'\n{subdir}: {len(npy_files)} token files')
        if npy_files:
            sample = np.load(str(npy_files[0]))
            print(f'  Sample shape: {sample.shape} (Q={sample.shape[0]}, T={sample.shape[1]})')
            print(f'  Duration: ~{sample.shape[1] / 12.5:.1f}s at 12.5Hz')
    else:
        print(f'\n{subdir}: [directory not found at {token_dir}]')

## Step 6: Pipeline Summary

Check all outputs and prepare for training.

In [ ]:
# Push pipeline output to HuggingFace Hub for persistence
from huggingface_hub import HfApi
import os

HF_REPO = "victormugambi/hibiki-sw-pipeline-output"  # change if needed
DIRECTION = f'{SOURCE_LANG}2{TARGET_LANG}'

source_subdir_map = {
    'common_voice': 'cv_sw',
    'kenspeech': 'kenspeech_sw',
    'directory': 'fleurs_en',
}
source_token_subdir = source_subdir_map.get(SOURCE, 'fleurs_en')

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not hf_token:
    print("WARNING: HF_TOKEN not set. Cannot push to HuggingFace.")
    print("Add your write-access token as Kaggle secret 'HF_TOKEN'.")
    print("Skipping push. Save the output manually via 'Save Version' in Kaggle.")
else:
    api = HfApi(token=hf_token)
    try:
        api.create_repo(repo_id=HF_REPO, repo_type="dataset", exist_ok=True, private=True)
        print(f"Repo ready: hf://datasets/{HF_REPO}")
    except Exception as e:
        if "403" in str(e) or "Forbidden" in str(e):
            print(f"ERROR: 403 Forbidden — HF_TOKEN lacks write access.")
        else:
            print(f"ERROR creating repo: {e}")
        hf_token = None

    if hf_token:
        print(f"Uploading {BASE_DIR} -> hf://datasets/{HF_REPO}/{DIRECTION}/")
        try:
            api.upload_folder(
                folder_path=BASE_DIR,
                repo_id=HF_REPO,
                repo_type="dataset",
                path_in_repo=DIRECTION,
                commit_message=f"Pipeline output: {DIRECTION} (source={SOURCE})",
                ignore_patterns=["*.pyc", "__pycache__"],
            )
            print(f"\nDone! https://huggingface.co/datasets/{HF_REPO}/tree/main/{DIRECTION}")
        except Exception as e:
            print(f"ERROR uploading: {e}")
            print("Data is still in /kaggle/working — use 'Save Version' to persist.")

# Summarize what was produced
print("\n=== Output Summary ===")
from pathlib import Path
for name, path in [
    ("Transcriptions", f"{BASE_DIR}/transcriptions/{SOURCE_LANG}"),
    ("Translations",   f"{BASE_DIR}/translations/{DIRECTION}"),
    ("Alignments",     f"{BASE_DIR}/alignments/{DIRECTION}"),
    ("Synthesized audio", f"{BASE_DIR}/synthetic_audio/{DIRECTION}"),
    ("Source tokens",  f"{BASE_DIR}/audio_tokens/{source_token_subdir}"),
    ("Synth tokens",   f"{BASE_DIR}/audio_tokens/synth_{TARGET_LANG}"),
]:
    if os.path.exists(path):
        n_json = len(list(Path(path).rglob('*.json')))
        n_wav  = len(list(Path(path).rglob('*.wav')))
        n_npy  = len(list(Path(path).rglob('*.npy')))
        parts = [f"{n} {ext}" for n, ext in [(n_json,'json'),(n_wav,'wav'),(n_npy,'npy')] if n]
        print(f"  {name}: {', '.join(parts) or '(empty)'}")
    else:
        print(f"  {name}: [NOT FOUND]")

## Step 7: Persist Output

Kaggle wipes `/kaggle/working` when the session ends. Save the pipeline output to HuggingFace Hub **before closing the session**.

In [ ]:
print('=' * 60)
print(f'PIPELINE COMPLETE: {SOURCE_LANG} -> {TARGET_LANG} (source={SOURCE})')
print('=' * 60)

source_subdir_map = {
    'common_voice': 'cv_sw',
    'kenspeech': 'kenspeech_sw',
    'directory': 'fleurs_en',
}
source_token_subdir = source_subdir_map.get(SOURCE, 'fleurs_en')

artifacts = {
    'Transcriptions': f'{BASE_DIR}/transcriptions/{SOURCE_LANG}',
    'Translations': f'{BASE_DIR}/translations/{DIRECTION}',
    'Alignments': f'{BASE_DIR}/alignments/{DIRECTION}',
    'Synthesized audio': SYNTH_DIR,
    'Source tokens': f'{AUDIO_TOKEN_DIR}/{source_token_subdir}',
    f'Target tokens (synth_{TARGET_LANG})': f'{AUDIO_TOKEN_DIR}/synth_{TARGET_LANG}',
}

for name, path in artifacts.items():
    if os.path.exists(path):
        n_json = len(list(Path(path).rglob('*.json')))
        n_wav = len(list(Path(path).rglob('*.wav')))
        n_npy = len(list(Path(path).rglob('*.npy')))
        parts = []
        if n_json: parts.append(f'{n_json} json')
        if n_wav: parts.append(f'{n_wav} wav')
        if n_npy: parts.append(f'{n_npy} npy')
        print(f'  {name}: {", ".join(parts)}')
    else:
        print(f'  {name}: [NOT FOUND]')

print(f'\n=== Next Steps ===')
if SOURCE_LANG == 'sw':
    print('1. (Optional) Re-run with SOURCE_LANG="en", TARGET_LANG="sw" for reverse direction')
else:
    print('1. Both directions done!')
print('2. Create S2ST manifest:')
print(f'   python data/prepare/create_s2st_manifest.py \\')
print(f'     --source synthetic \\')
print(f'     --source_token_dir {AUDIO_TOKEN_DIR}/{source_token_subdir} \\')
print(f'     --target_token_dir {AUDIO_TOKEN_DIR}/synth_{TARGET_LANG} \\')
print(f'     --translation_dir {TRANSLATION_DIR} \\')
print(f'     --text_token_dir {BASE_DIR}/aligned_text/{DIRECTION} \\')
print(f'     --output_manifest {BASE_DIR}/manifests/{DIRECTION}_train.tsv \\')
print(f'     --tokenizer_model <path>/sp_ensw_32k.model \\')
print(f'     --direction {DIRECTION}')
print('3. Start training (Stages 1-4)')